# 10 — IoU Threshold Sensitivity (DIAGNOSTIC)

> **Temporary diagnostic tool — not part of the final pipeline.**

Compares city-level F1 at two IoU match thresholds computed from stored outputs:

| Threshold | How computed |
|-----------|-------------|
| **τ = 0.50** | Exact — from stored tile-level tp/fp/fn (matches pipeline) |
| **τ = 0.75** | Exact — by filtering stored match parquets to IoU ≥ 0.75 |

**Output:** `outputs/scratch/iou_threshold_sensitivity.csv`

In [ ]:
!pip install -q scikit-posthocs
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Cell 1 — Setup & load baseline ───────────────────────────────────────────
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

CONFIG_PATH  = Path('/content/drive/MyDrive/WorldBank/FY26 - DEP/Gates Foundation/Building Dataset Validation/configs/validation_configs.yaml')
PROJECT_ROOT = CONFIG_PATH.parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

METRICS_ROOT = PROJECT_ROOT / 'outputs' / 'metrics'
SCRATCH_DIR  = PROJECT_ROOT / 'outputs' / 'scratch'
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)

TILE_SENTINEL  = 'vector_metrics_tiles_all_datasets.parquet'
MATCH_SENTINEL = 'vector_matches_all_datasets.parquet'

# ── SpaceNet7 flag ────────────────────────────────────────────────────────────
TRACKER_PATH = PROJECT_ROOT / 'data/02_interim/aoi_tracker.csv'
ref_source_map = {}
if TRACKER_PATH.exists():
    tracker = pd.read_csv(TRACKER_PATH, dtype=str)
    tracker.columns = tracker.columns.str.strip()
    if 'reference_source' in tracker.columns and 'dataset_folder_name' in tracker.columns:
        ref_source_map = (
            tracker.dropna(subset=['dataset_folder_name'])
            .set_index('dataset_folder_name')['reference_source']
            .str.strip().str.lower()
            .to_dict()
        )
        sn7 = sum(1 for v in ref_source_map.values() if v == 'spacenet')
        print(f'Tracker loaded: {len(ref_source_map)} cities  ({sn7} SpaceNet7)')
    else:
        print('[WARN] reference_source column not found — all cities treated as non-SpaceNet')
else:
    print(f'[WARN] Tracker not found at {TRACKER_PATH}')

print('Cell 1 done.')

In [ ]:
# ── Cell 2 — Compute F1 at τ=0.50 and τ=0.75 for every city × dataset ────────

all_rows = []
city_dirs = sorted(p.parent for p in METRICS_ROOT.rglob(TILE_SENTINEL))
print(f'Processing {len(city_dirs)} cities...')

for city_dir in city_dirs:
    city = city_dir.name

    # τ = 0.50 — exact from stored tile tp/fp/fn
    try:
        tile_df = pd.read_parquet(city_dir / TILE_SENTINEL)
    except Exception as e:
        print(f'  [WARN] {city}: {e}')
        continue

    f1_050 = {}
    for ds, g in tile_df.groupby('dataset'):
        tp = int(g['tp'].sum()); fp = int(g['fp'].sum()); fn = int(g['fn'].sum())
        p  = tp / (tp + fp) if (tp + fp) else 0.0
        r  = tp / (tp + fn) if (tp + fn) else 0.0
        f1_050[ds] = round(2*p*r / (p+r) if (p+r) else 0.0, 4)

    # τ = 0.75 — filter stored matches to IoU ≥ 0.75
    match_path = city_dir / MATCH_SENTINEL
    f1_075 = {}
    if match_path.exists():
        try:
            matches_df = pd.read_parquet(match_path)
            for ds, g in tile_df.groupby('dataset'):
                n_ref  = int(g['n_ref'].sum())
                n_cand = int(g['n_cand'].sum())
                tp75   = int((matches_df[matches_df['dataset'] == ds]['iou'] >= 0.75).sum())
                fp75   = max(0, n_cand - tp75)
                fn75   = max(0, n_ref  - tp75)
                p  = tp75 / (tp75 + fp75) if (tp75 + fp75) else 0.0
                r  = tp75 / (tp75 + fn75) if (tp75 + fn75) else 0.0
                f1_075[ds] = round(2*p*r / (p+r) if (p+r) else 0.0, 4)
        except Exception as e:
            print(f'  [WARN] {city}: matches error: {e}')

    is_sn7 = (ref_source_map.get(city, 'other') == 'spacenet')
    for ds in tile_df['dataset'].unique():
        all_rows.append({
            'city':        city,
            'dataset':     ds,
            'is_spacenet7': is_sn7,
            'f1_iou50':    f1_050.get(ds, float('nan')),
            'f1_iou75':    f1_075.get(ds, float('nan')),
        })

df_sensitivity = pd.DataFrame(all_rows)
df_sensitivity['delta_iou75_vs_50'] = (
    df_sensitivity['f1_iou75'] - df_sensitivity['f1_iou50']
).round(4)

print(f'\nDone: {df_sensitivity["city"].nunique()} cities × '
      f'{df_sensitivity["dataset"].nunique()} datasets = {len(df_sensitivity)} rows')
print('\nSample:')
display(df_sensitivity.head(6))

In [ ]:
# ── Cell 3 — Summary table + box plots ───────────────────────────────────────

GROUP_PALETTE = {'SpaceNet7': '#0072B2', 'Non-SpaceNet': '#E69F00'}
group_labels  = {True: 'SpaceNet7', False: 'Non-SpaceNet'}

df_city = (
    df_sensitivity
    .groupby(['city', 'is_spacenet7'])[['f1_iou50', 'f1_iou75', 'delta_iou75_vs_50']]
    .mean()
    .reset_index()
    .round(4)
)
df_city['group'] = df_city['is_spacenet7'].map(group_labels)

print('=== City-level summary (mean F1 across datasets) ===')
print(f'Total cities: {len(df_city)}  '
      f'(SpaceNet7: {int(df_city["is_spacenet7"].sum())}  |  '
      f'Non-SpaceNet: {int((~df_city["is_spacenet7"]).sum())})')
print()
for is_sn, grp in df_city.groupby('is_spacenet7'):
    label = 'SpaceNet7' if is_sn else 'Non-SpaceNet'
    print(f'  {label} ({len(grp)} cities):')
    for col in ['f1_iou50', 'f1_iou75', 'delta_iou75_vs_50']:
        print(f'    {col:<25}  mean={grp[col].mean():.4f}  '
              f'median={grp[col].median():.4f}  std={grp[col].std():.4f}')
    print()

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: F1 at τ=0.50 vs τ=0.75 side by side
df_melt = df_city.melt(
    id_vars=['city', 'group'],
    value_vars=['f1_iou50', 'f1_iou75'],
    var_name='threshold', value_name='f1'
)
df_melt['threshold_label'] = df_melt['threshold'].map(
    {'f1_iou50': 'τ = 0.50', 'f1_iou75': 'τ = 0.75'})

sns.boxplot(data=df_melt, x='threshold_label', y='f1', hue='group',
            palette=GROUP_PALETTE, ax=axes[0], linewidth=1.2)
axes[0].set_title('F1 at τ = 0.50 vs τ = 0.75', fontweight='bold')
axes[0].set_xlabel('IoU threshold')
axes[0].set_ylabel('City-level F1')
axes[0].set_ylim(0, 1.05)
axes[0].legend(title='Group', fontsize=8)

# Panel 2: Δ F1 (τ=0.75 minus τ=0.50) — how much stricter matching costs
sns.boxplot(data=df_city, x='group', y='delta_iou75_vs_50',
            palette=GROUP_PALETTE, ax=axes[1], linewidth=1.2)
axes[1].axhline(0, color='grey', linestyle='--', linewidth=1)
axes[1].set_title('Δ F1: τ=0.75 minus τ=0.50\n(negative = stricter threshold costs F1)',
                  fontweight='bold')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('ΔF1')

# Panel 3: Scatter τ=0.50 vs τ=0.75 per dataset
DATASET_MARKERS = {'overture': 'o', 'gba': 's', 'globfp': '^'}
for ds, ds_grp in df_sensitivity.groupby('dataset'):
    ds_grp = ds_grp.copy()
    ds_grp['group'] = ds_grp['is_spacenet7'].map(group_labels)
    for grp_label, g in ds_grp.groupby('group'):
        axes[2].scatter(
            g['f1_iou50'], g['f1_iou75'],
            color=GROUP_PALETTE[grp_label],
            marker=DATASET_MARKERS.get(ds, 'o'),
            alpha=0.4, s=12,
            label=f'{ds} / {grp_label}' if grp_label == list(GROUP_PALETTE.keys())[0] else None
        )
axes[2].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='y = x')
axes[2].set_xlabel('F1 at τ = 0.50')
axes[2].set_ylabel('F1 at τ = 0.75')
axes[2].set_title('Scatter: τ=0.50 vs τ=0.75\n(below y=x = stricter threshold reduces F1)',
                  fontweight='bold')

for ax in axes:
    ax.grid(axis='y', alpha=0.3)
    sns.despine(ax=ax)

fig.suptitle('IoU threshold sensitivity — τ=0.50 (pipeline default) vs τ=0.75',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout()
out_fig = SCRATCH_DIR / 'iou_threshold_sensitivity_boxplot.png'
fig.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out_fig}')

In [ ]:
# ── Cell 4 — Save CSV ─────────────────────────────────────────────────────────
out_path = SCRATCH_DIR / 'iou_threshold_sensitivity.csv'
df_sensitivity.to_csv(out_path, index=False)

print(f'Saved → {out_path}')
print(f'  {len(df_sensitivity):,} rows × {len(df_sensitivity.columns)} columns')
print(f'  Columns: {list(df_sensitivity.columns)}')
print()
print('=== Global means by dataset ===')
display(
    df_sensitivity
    .groupby('dataset')[['f1_iou50', 'f1_iou75', 'delta_iou75_vs_50']]
    .mean()
    .round(4)
)